# Circuit Tracer Analysis Backend Demo

This notebook demonstrates the **Interpretune analysis ops pipeline** for circuit-tracer analysis,
running the full **semantic concept intervention** flow as a single composite pipeline:

- **Concept Direction** — Compute a directional concept vector (e.g. "Capitals − States")
- **Attribution Graph** — Generate a circuit-level attribution graph from the concept direction
- **Node Influence** — Score graph nodes by their causal influence on the concept
- **Top Features** — Extract the most influential transcoder features
- **Feature Intervention** — Amplify those features and measure how predictions shift

The five ops above are composed into the `intervention_from_concept` pipeline, which chains
them automatically and returns results in a single `AnalysisBatch`.

> **Tip:** For individual op dispatching and the `DISPATCHER` API, see the
> [op_collection_example](../example_op_collections/op_collection_example.ipynb) notebook.

The `backend` parameter selects the circuit-tracer backend: **NNsight** (default) or
**TransformerLens** — both are validated by the parameterized notebook tests. (The
TransformerLens backend uses the legacy `HookedTransformer` path; circuit-tracer does not yet
support `TransformerBridge` — see the tracking notes in `docs/circuit_tracer_backend_support.md`.)


In [ ]:
# @title Imports { display-mode: "form" }
from pprint import pformat

import interpretune as it
from it_examples import _ACTIVE_PATCHES  # noqa: F401
from it_examples.example_module_registry import MODULE_EXAMPLE_REGISTRY
from it_examples.utils.example_helpers import required_os_env
from it_examples.utils.nb_ui_utils import (
    display_target_gap,
    display_top_features_comparison,
    display_topk_token_predictions,
    resolve_feature_explanations,
)
from interpretune import ITSession, ITSessionConfig
from interpretune.analysis.ops.base import AnalysisBatch
from interpretune.config import AnalysisCfg, init_analysis_cfgs

## Notebook Parameters

This cell contains parameters that can be injected by papermill during parameterized test runs.

`dashboard_mode` selects where feature-dashboard links point: `"public"` uses
[neuronpedia.org](https://www.neuronpedia.org), `"local"` uses a local Neuronpedia dev webapp
(`local_webapp_url`) — see the Neuronpedia
[localhost guide](https://github.com/hijohnnylin/neuronpedia?tab=readme-ov-file#localhost-installation).


In [ ]:
# Parameters - These will be injected by papermill during parameterized test runs
backend = "nnsight"  # Options: "transformerlens", "nnsight"
core_log_dir = None  # Directory to save analysis logs (if None, a temp directory will be created)
intervention_scale_factor = 10.0  # Multiplier for feature intervention amplitudes
dashboard_mode = "public"  # Options: "public" (neuronpedia.org links), "local" (local Neuronpedia dev webapp)
local_webapp_url = "http://localhost:3000"  # Feature-dashboard base URL used when dashboard_mode="local"

In [ ]:
# @title Environment Setup { display-mode: "form" }
env_path: str | None = None  # set to '/full/path/to/.env' to override
os_env_reqs = None
assert required_os_env(env_path=env_path, env_reqs=os_env_reqs)

## Session Configuration

We load a registered example module configuration for Gemma-2-2b with the RTE task,
then configure it for the selected backend. The `"gemma"` transcoder set uses Gemma Scope
transcoders, matching the upstream
[attribution_targets_demo](https://github.com/safety-research/circuit-tracer/blob/main/demos/attribution_targets_demo.ipynb).

> **Note:** the first `MODULE_EXAMPLE_REGISTRY` access hydrates *every* example registry entry.
> Per-entry config-normalization feedback (categorized as `ITInstantiationFeedbackWarning`) is
> suppressed during that bulk hydration so this cell only surfaces messages relevant to the
> requested configuration; directly instantiating a config still shows its feedback.


In [ ]:
# Load the demo configuration from the example module registry
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = MODULE_EXAMPLE_REGISTRY.get("gemma2.rte_demo.circuit_tracer")

# Configure backend
base_it_cfg.circuit_tracer_cfg.backend = backend
print(f"Backend: {backend}")

# Optionally override core_log_dir
if core_log_dir:
    base_it_cfg.core_log_dir = core_log_dir

# Use Gemma Scope transcoders (matches upstream attribution_targets_demo)
base_it_cfg.circuit_tracer_cfg.transcoder_set = "gemma"

# Configure intervention settings
base_it_cfg.circuit_tracer_cfg.intervention_value_source = "top_feature_activation_values"
base_it_cfg.circuit_tracer_cfg.intervention_scale_factor = intervention_scale_factor

print(pformat(base_it_cfg.circuit_tracer_cfg))

In [ ]:
# Configure the session with the appropriate adapter composition
if backend == "nnsight":
    adapter_ctx = (it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer)
else:
    # the TL circuit-tracer backend needs the transformer_lens adapter in the composition
    # (it provides the replacement-model init path; see docs/circuit_tracer_backend_support.md)
    adapter_ctx = (it.Adapter.core, it.Adapter.transformer_lens, it.Adapter.circuit_tracer)

session_cfg = ITSessionConfig(
    adapter_ctx=adapter_ctx,
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)

it_session = ITSession(session_cfg)

# Initialize session (loads model, sets up hooks, etc.)
it.it_init(**it_session)

# Set up analysis config on module
module = it_session.module
tokenizer = module.replacement_model.tokenizer

graph_op = it.compute_attribution_graph
module.analysis_cfg = AnalysisCfg(target_op=graph_op, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])

print(f"Module type: {type(module).__name__}")
print("Session initialized successfully!")

## Analysis Pipeline

We use the `intervention_from_concept` composite op to chain all five analysis operations
in a single pipeline call. Virtual logit target IDs (from concept-direction attribution
targets) are automatically resolved to real vocabulary token IDs.

| Step | Op | Purpose |
|------|----|---------|
| Pipeline | `concept_direction` | Compute a directional concept vector via paired rejection |
| Pipeline | `compute_attribution_graph` | Generate a circuit-level attribution graph for the prompt |
| Pipeline | `graph_node_influence` | Score graph nodes by causal influence on the concept |
| Pipeline | `extract_top_features` | Rank and extract the most influential transcoder features |
| Pipeline | `feature_intervention_forward` | Amplify top features and measure prediction shifts |

We define the concept groups, prompt, and key tokens, then run the pipeline.

In [ ]:
# Concept groups (SentencePiece tokens with leading ▁)
capitals = ["▁Austin", "▁Sacramento", "▁Olympia", "▁Atlanta"]
states = ["▁Texas", "▁California", "▁Washington", "▁Georgia"]
concept_label = "Concept: Capitals − States"

# Analysis prompt
prompt = "Fact: the capital of the state containing Dallas is"

# Key tokens for result comparison (matching upstream attribution_targets_demo)
austin_id = tokenizer.encode("▁Austin", add_special_tokens=False)[-1]
dallas_id = tokenizer.encode("▁Dallas", add_special_tokens=False)[-1]
key_tokens = [("Austin", austin_id), ("Dallas", dallas_id)]

print(f"Prompt: '{prompt}'")
print(f"Key tokens: Austin={austin_id}, Dallas={dallas_id}")

In [ ]:
# Run the full intervention_from_concept pipeline (all 5 ops in one call)
full_pipeline = it.intervention_from_concept
print(f"Full pipeline: {' → '.join(op.name for op in full_pipeline.composition)}")

results = full_pipeline(
    module,
    AnalysisBatch(
        concept_group_a=capitals,
        concept_group_b=states,
        concept_label=concept_label,
        concept_direction_mode="paired_rejection",
        prompts=[prompt],
    ),
    None,
    0,
    top_n=10,
    intervention_scale_factor=intervention_scale_factor,
)
print("Pipeline complete!")

## Results: Concept Direction

The concept direction vector captures the "capital-ness" concept via paired rejection,
projecting out state-specific components from each capital embedding.

The reported **direction norm** should read `1.0000`: `concept_direction` returns a
unit-normalized vector, so the print is a cheap sanity check —
a `0`/`nan` norm flags a degenerate concept pair (groups canceling under paired rejection), and
any non-unit value flags an aggregation/normalization regression. Because the direction is unit-norm, the applied
perturbation magnitude in direct `add`-mode steering is exactly `direction_scale_factor`, and
the attribution/node-influence rankings below are unaffected by direction scale.


In [ ]:
print(f"Concept direction shape: {results.concept_direction.shape}")
print(f"Concept label: {results.concept_label}")
direction_norm = float(results.concept_direction.norm())
print(f"Direction norm: {direction_norm:.4f}")
assert abs(direction_norm - 1.0) < 1e-3, "concept_direction should return a unit-normalized vector"

## Results: Top Features

The top transcoder features ranked by node influence scores.
Each feature is a ``(layer, position, feature_index)`` tuple.

> For interactive node influence visualization, see the
> [attribution_analysis](../../analysis_injection/attribution_analysis.ipynb) notebook.

In [ ]:
# Build feature tuples and scores for display
features = [tuple(f.tolist()) for f in results.top_feature_ids]
scores = results.top_feature_scores.tolist()

# Feature-dashboard links: public neuronpedia.org or a local Neuronpedia dev webapp
neuronpedia_base_url = "https://www.neuronpedia.org" if dashboard_mode == "public" else local_webapp_url

# Best-effort explanation text from the feature API (public or local webapp); unmapped features
# simply render an empty Explanation cell
feature_explanations = resolve_feature_explanations(
    model_id="gemma-2-2b",
    source_set="gemmascope-transcoder-16k",
    feature_tuples=[(f[0], f[-1]) for f in features],
    base_url=neuronpedia_base_url,
)

display_top_features_comparison(
    {"Top Features (by node influence)": features},
    {"Top Features (by node influence)": scores},
    neuronpedia_model="gemma-2-2b",
    neuronpedia_base_url=neuronpedia_base_url,
    feature_explanations=feature_explanations,
)

## Results: Feature Intervention

Amplifying the top features by the configured scale factor and measuring how
predictions shift. If the pipeline correctly identifies "capital" features,
amplifying them should increase the probability of **Austin** relative to **Dallas**.

In [ ]:
pre_logits = results.pre_intervention_logits.float().cpu()
post_logits = results.post_intervention_logits.float().cpu()

# Rich pre/post comparison with key tokens
display_topk_token_predictions(
    prompt,
    pre_logits,
    post_logits,
    tokenizer,
    k=5,
    key_tokens=key_tokens,
)

# Consolidated pre/post probabilities, logits, and the Austin - Dallas gap in one table
pre_gap, post_gap = display_target_gap(
    pre_logits,
    post_logits,
    ("Austin", austin_id),
    ("Dallas", dallas_id),
    title="Feature intervention — Austin vs Dallas",
)

# Sanity gate (both backends): amplifying "capital" features must widen the Austin-Dallas gap
assert post_gap > pre_gap, "feature intervention should widen the Austin-Dallas logit gap"

## Summary

The `intervention_from_concept` pipeline ran all five analysis ops in a single call:

| Op | Result |
|----|---------|
| `concept_direction` | Computed a "Capitals − States" concept vector via paired rejection |
| `compute_attribution_graph` | Generated a circuit-level attribution graph for the prompt |
| `graph_node_influence` | Scored graph nodes by causal influence |
| `extract_top_features` | Ranked and extracted the top-10 most influential features |
| `feature_intervention_forward` | Amplified top features and measured prediction shifts |

Virtual logit target IDs from the attribution graph are automatically resolved to real
vocabulary token IDs, so no manual override is needed.

**Key result:** Amplifying the identified "capital" features shifts the model's prediction
towards the correct answer (Austin), validating that the circuit-tracer pipeline identifies
causally relevant features.

### What's Next

- **Concept-Direction Steering Demo**: See `ct_concept_steering_demo.ipynb` for store- and embed-based,
  sign-aware multi-feature concept steering with public or locally served feature dashboards
- **RTE research direction**: RTE-focused concept-direction research continues in
  [interpretune#220](https://github.com/speediedan/interpretune/issues/220)
- **Analysis Injection**: See `attribution_analysis.ipynb` for node influence visualization via injection hooks
- **Op Dispatching**: See `op_collection_example.ipynb` for the `DISPATCHER` API and individual op usage
- **AnalysisRunner**: For batch analysis workflows, use `AnalysisRunner` with `AnalysisCfg` objects
- **Basic CT Tutorial**: See `circuit_tracer_adapter_example_basic.ipynb` for the foundational adapter tutorial